# Restructuration

Une fois toutes les données téléchargées, certaines restructurations sont nécessaires. 

Voir le [README](./README.md) pour plus de détails.

In [3]:
import pandas as pd

### GCPE IFT : hétérogénéïté entre 2017 et 2021

Le fichier obtenu pour l'enquête PK 2017 expose uniquement la colonne `Orge` alors que celui de l'enquête PK 2021 expose à la fois `Orge de printemps` et `Orge d'hiver`.

Pour fusionner `Orge de printemps` et `Orge d'hiver` de façon cohérente, on a besoin de la surface développée de chacune de ces cultures en 2021.

On modifie donc ici : 
- `ift_culture_ancienne_region_gcpe_2021.csv` -> fusion des colonnes d'orge. 
- `surface_espece_ancienne_region.csv` -> fusion des colonnes d'orge.

> Attention, pour certaines lignes, on a pas de valeur d'IFT pour l'Orge de printemps. Dans ces cas là, on considéreras que l'IFT Orge total correspondra à l'IFT Orge d'hiver.

In [4]:
# chargement des données
df = {}
df['ift_culture_ancienne_region_gcpe_2017'] = pd.read_csv('./ift/gcpe/ift_culture_ancienne_region_gcpe_2017.csv')
df['ift_culture_ancienne_region_gcpe_2021'] = pd.read_csv('./ift/gcpe/ift_culture_ancienne_region_gcpe_2021.csv')
df['surface_espece_ancienne_region'] = pd.read_csv('./surface/gcpe/surface_espece_ancienne_region.csv')

In [5]:
df['surface_espece_ancienne_region_2021_orge_printemps'] = df['surface_espece_ancienne_region'].loc[
    (df['surface_espece_ancienne_region']['Campagne'] == 2021) &
    (df['surface_espece_ancienne_region']['Espece_SSP'] == 'Orge de printemps')
].set_index('Espece_SSP')[['Nom_Ancienne_Region', 'Surface_Espece_Region']]

df['surface_espece_ancienne_region_2021_orge_hiver'] = df['surface_espece_ancienne_region'].loc[
    (df['surface_espece_ancienne_region']['Campagne'] == 2021) &
    (df['surface_espece_ancienne_region']['Espece_SSP'] == "Orge d'hiver")
].set_index('Espece_SSP')[['Nom_Ancienne_Region', 'Surface_Espece_Region']]

# ajout de la surface de l'orge d'hiver
left = df['ift_culture_ancienne_region_gcpe_2021'] 
right = df['surface_espece_ancienne_region_2021_orge_hiver'].rename(columns={
    'Surface_Espece_Region' : 'surface_orge_hiver'
})
df['ift_culture_ancienne_region_gcpe_2021_extanded'] = pd.merge(
    left, 
    right, 
    left_on = ['nom_ancienne_region'], 
    right_on = ['Nom_Ancienne_Region']
)

# ajout de la surface de l'orge de printemps
left = df['ift_culture_ancienne_region_gcpe_2021_extanded'] 
right = df['surface_espece_ancienne_region_2021_orge_printemps'].rename(columns={
    'Surface_Espece_Region' : 'surface_orge_printemps'
})
df['ift_culture_ancienne_region_gcpe_2021_extanded'] = pd.merge(
    left, 
    right, 
    left_on = ['nom_ancienne_region'], 
    right_on = ['Nom_Ancienne_Region']
)

# calcul surface totale orge
df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge'] = \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge_hiver'] + \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge_printemps']

# identification des cas où on a pas d'ift orge printemps
region_sans_orge_printemps = df['ift_culture_ancienne_region_gcpe_2021_extanded'].loc[
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['Orge de printemps'].isna()
].index

# calcul des proportions
df['ift_culture_ancienne_region_gcpe_2021_extanded']['proportion_orge_hiver'] = \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge_hiver'] / \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge']


df['ift_culture_ancienne_region_gcpe_2021_extanded']['proportion_orge_printemps'] = \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge_printemps'] / \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['surface_orge']

# calcul de l'IFT orge total
df['ift_culture_ancienne_region_gcpe_2021_extanded']['Orge'] = \
    round(df['ift_culture_ancienne_region_gcpe_2021_extanded']['proportion_orge_printemps']*df['ift_culture_ancienne_region_gcpe_2021_extanded']["Orge de printemps"] + \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']['proportion_orge_hiver']*df['ift_culture_ancienne_region_gcpe_2021_extanded']["Orge d'hiver"], 2)

# correction dans le cas où IFT orge printemps non accessible
df['ift_culture_ancienne_region_gcpe_2021_extanded'].loc[
    region_sans_orge_printemps
    , 'Orge'] = \
    df['ift_culture_ancienne_region_gcpe_2021_extanded']["Orge d'hiver"]

# correction de surface_espece_ancienne_region (fusion de Orge de printemps et Orge)
df['surface_espece_ancienne_region']['Espece_SSP_corrige'] = \
    df['surface_espece_ancienne_region']['Espece_SSP']

df['surface_espece_ancienne_region'].loc[
    df['surface_espece_ancienne_region']['Espece_SSP_corrige'].isin([
        "Orge de printemps", "Orge d'hiver"
    ]), 'Espece_SSP_corrige'
] = "Orge"

df['surface_espece_ancienne_region_corrige'] = df['surface_espece_ancienne_region'].groupby([
    "Espece_SSP_corrige", "Campagne", "Nom_Ancienne_Region"
]).agg({
    'Surface_Espece_Region' : 'sum',
    'Surface_Region' : 'first', 
    'Part_surface_espece_region' : 'sum'
}).reset_index().sort_values([
    'Nom_Ancienne_Region', 'Campagne', 'Espece_SSP_corrige'
])

# export des informations
df['surface_espece_ancienne_region_corrige'].to_csv('./restructure/surface_espece_ancienne_region_restructure.csv', index=False)
df['ift_culture_ancienne_region_gcpe_2021_extanded'][
    list(df['ift_culture_ancienne_region_gcpe_2021'].columns) + ['Orge']
].to_csv('./restructure/ift_culture_ancienne_region_gcpe_2021_restructure.csv', index=False)

In [6]:
df['ift_culture_ancienne_region_gcpe_2021_extanded']

,nom_ancienne_region,Blé tendre,Blé dur,Triticale,Colza,Tournesol,Pois protéagineux,Maïs fourrage,Maïs grain,Betterave sucrière,...,Orge de printemps,Orge d'hiver,Nom_Ancienne_Region_x,surface_orge_hiver,Nom_Ancienne_Region_y,surface_orge_printemps,surface_orge,proportion_orge_hiver,proportion_orge_printemps,Orge
0,Alsace,3.4,NaN,NaN,NaN,NaN,NaN,NaN,2.8,NaN,...,NaN,NaN,Alsace,5450,Alsace,600,6050,0.900826,0.099174,NaN
1,Aquitaine,3.4,NaN,1.8,NaN,2.7,NaN,2.1,2.9,NaN,...,NaN,2.3,Aquitaine,16605,Aquitaine,2961,19566,0.848666,0.151334,2.30
2,Auvergne,3.9,NaN,2.6,6.1,2.4,NaN,2.2,3.6,NaN,...,NaN,3.5,Auvergne,23340,Auvergne,2040,25380,0.919622,0.080378,3.50
3,Basse-Normandie,5.3,NaN,3.4,6.8,NaN,5.3,2.3,NaN,NaN,...,2.7,4.6,Basse-Normandie,50310,Basse-Normandie,8200,58510,0.859853,0.140147,4.33
4,Bourgogne,4.4,NaN,2.7,6.7,2.2,3.9,2.4,2.8,NaN,...,2.8,4.8,Bourgogne,120100,Bourgogne,53400,173500,0.692219,0.307781,4.18
5,Bretagne,4.5,NaN,3.6,3.7,NaN,NaN,2.4,2.6,NaN,...,2.4,4.3,Bretagne,76085,Bretagne,18275,94360,0.806327,0.193673,3.93
6,Centre,5.9,5.7,2.8,6.8,2.9,5.0,2.6,3.4,7.7,...,3.8,5.5,Centre,201575,Centre,77045,278620,0.723476,0.276524,5.03
7,Champagne-Ardenne,5.4,NaN,2.6,6.5,2.1,4.6,2.6,3.0,7.4,...,3.5,5.1,Champagne-Ardenne,111930,Champagne-Ardenne,155850,267780,0.417992,0.582008,4.17
8,Franche-Comté,4.5,NaN,2.8,6.9,NaN,NaN,2.2,2.6,NaN,...,NaN,4.3,Franche-Comté,26450,Franche-Comté,2630,29080,0.909560,0.090440,4.30
9,Guadeloupe,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,Guadeloupe,0,Guadeloupe,0,0,NaN,NaN,NaN
